In [ ]:

print("QUANTIZED GAME DIALOGUE SIMULATOR")
print("---------------------------------")
print("Welcome to our AI Game Dialogue Project!")
print("We will train a small GPT model.")
print("Then we will apply quantization.")


In [ ]:



import os


os.makedirs("dataset", exist_ok=True)


dialogue = """
Player: Hello, who are you?
Wizard: I am the ancient wizard of this kingdom.

Player: Can you help me?
Wizard: Yes, young traveler. What do you need?

Player: Where is the dragon?
Wizard: The dragon lives inside the dark mountain.

Player: How can I defeat the dragon?
Wizard: You need a magical sword to defeat it.

Player: Where can I find the sword?
Wizard: The sword is hidden inside the ancient castle.

Player: Who guards the castle?
Knight: I guard the castle. Prove your courage.

Player: I am ready for the challenge.
Knight: Then show me your strength.

Player: Where is the village?
Villager: The village is beyond the green forest.

Player: Is the forest dangerous?
Villager: Yes, many monsters live there.

Player: I found a treasure chest.
Wizard: Excellent! You have discovered ancient gold.

Player: I am afraid of the dragon.
Wizard: Courage is stronger than fear.

Player: The dragon is attacking!
Knight: Take cover! I will protect you.

Player: I defeated the dragon!
Villager: You are the hero of our kingdom!

Player: What should I do now?
Wizard: Your next adventure awaits you.

Dragon: Who dares to enter my mountain?
Player: I have come to defeat you.

Dragon: You cannot defeat me!
Player: I will never give up.

Wizard: Welcome back, brave hero.
Player: Thank you for helping me.

Knight: The kingdom is safe again.
Villager: Long live the hero!
"""


file_path = "dataset/game_dialogue.txt"

with open(file_path, "w", encoding="utf-8") as file:
    file.write(dialogue)

print("Dataset created successfully!")
print("File location:", file_path)
print("Total characters:", len(dialogue))


In [ ]:

# Read and display our dataset

with open("dataset/game_dialogue.txt", "r", encoding="utf-8") as file:
    content = file.read()

print(content[:500])


In [ ]:
# Install required libraries (Google Colab/Jupyter)
%pip install -q torch numpy matplotlib pandas

print("Libraries installed successfully!")


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Selected device:", device)
print("Libraries imported successfully!")


In [ ]:

# Load game dialogue dataset

with open("dataset/game_dialogue.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Create character vocabulary
chars = sorted(list(set(text)))

vocab_size = len(chars)

# Character encoding and decoding
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Convert text into numbers
data = torch.tensor(encode(text), dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Total training characters:", len(data))
print("Sample encoded data:", data[:20].tolist())


In [ ]:

# Split dataset into training and validation sets

train_size = int(0.9 * len(data))

train_data = data[:train_size]
val_data = data[train_size:]

print("Training data size:", len(train_data))
print("Validation data size:", len(val_data))


In [ ]:

# GPT model configuration

batch_size = 16
block_size = 64
max_iters = 1000
learning_rate = 0.001

n_embd = 64
n_head = 4
n_layer = 2
dropout = 0.1

print("GPT configuration completed!")
print("Embedding dimension:", n_embd)
print("Attention heads:", n_head)
print("Transformer layers:", n_layer)


In [ ]:

# Generate training batches

def get_batch(split):
    source = train_data if split == "train" else val_data

    if len(source) <= block_size:
        raise ValueError(f"{split} data must contain more than {block_size} characters.")
    ix = torch.randint(0, len(source) - block_size, (batch_size,))

    x = torch.stack([
        source[i:i + block_size] for i in ix
    ])

    y = torch.stack([
        source[i + 1:i + block_size + 1] for i in ix
    ])

    return x.to(device), y.to(device)

xb, yb = get_batch("train")

print("Input batch shape:", xb.shape)
print("Target batch shape:", yb.shape)
print("Batch generation successful!")


In [ ]:

# GPT MODEL: SELF ATTENTION

class Head(nn.Module):
    """Single self-attention head"""

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        weights = q @ k.transpose(-2, -1)
        weights = weights * (k.shape[-1] ** -0.5)

        weights = weights.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        weights = F.softmax(weights, dim=-1)
        weights = self.dropout(weights)

        v = self.value(x)
        output = weights @ v

        return output

print("Self-attention head created successfully!")


In [ ]:

# MULTI-HEAD ATTENTION

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        output = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        output = self.dropout(self.proj(output))
        return output


# FEED FORWARD NETWORK

class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


# TRANSFORMER BLOCK

class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

print("Transformer blocks created successfully!")


In [ ]:

# COMPLETE GPT LANGUAGE MODEL

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        token_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = token_emb + pos_emb

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, loss = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx


model = GPTLanguageModel().to(device)

print("GPT model created successfully!")

print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters())
)


In [ ]:

# GPT MODEL TRAINING

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

# Training loop
for step in range(max_iters):

    # Evaluate training batch
    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    # Display progress
    if step % 100 == 0:
        print(
            f"Step: {step}, Loss: {loss.item():.4f}"
        )

print("\nGPT training completed successfully!")


In [ ]:

# SAVE TRAINED MODEL

torch.save(model.state_dict(), "game_gpt_model.pth")

print("Trained GPT model saved successfully!")


In [ ]:

# GENERATE GAME DIALOGUE

context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=300
)

print("Generated Game Dialogue:")
print("------------------------")
print(decode(generated[0].tolist()))


In [ ]:

import os

# Original model size
original_size = os.path.getsize("game_gpt_model.pth") / 1024

print("Original GPT Model Size:", round(original_size, 2), "KB")


In [ ]:

# Create a CPU copy of the trained model

model_cpu = GPTLanguageModel()

model_cpu.load_state_dict(
    torch.load(
        "game_gpt_model.pth",
        map_location="cpu",
        weights_only=True
    )
)

model_cpu.eval()

print("CPU model prepared successfully!")


In [ ]:
# DYNAMIC POST-TRAINING QUANTIZATION

import torch.ao.quantization as quant

# Dynamic quantization is supported on CPU. Quantize Linear layers to int8.
quantized_dynamic_model = quant.quantize_dynamic(
    model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)
quantized_dynamic_model.eval()

print("Dynamic quantization completed successfully!")


In [ ]:

# STEP 20: Corrected Static Quantization

import torch
import torch.nn as nn
import torch.ao.quantization as quant

# Select quantization backend
available_backends = [
    name for name in ("x86", "fbgemm", "qnnpack")
    if name in torch.backends.quantized.supported_engines
]
if not available_backends:
    raise RuntimeError("This PyTorch build has no supported quantized CPU backend.")
backend = "x86" if "x86" in available_backends else available_backends[0]
torch.backends.quantized.engine = backend

# Load original trained model
static_model = GPTLanguageModel()

static_model.load_state_dict(
    torch.load(
        "game_gpt_model.pth",
        map_location="cpu",
        weights_only=True
    )
)

static_model.eval()

# Create wrapper for static quantization
class StaticQuantLinear(nn.Module):
    def __init__(self, linear):
        super().__init__()
        self.quant = quant.QuantStub()
        self.linear = linear
        self.dequant = quant.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.linear(x)
        x = self.dequant(x)
        return x

# Configure quantization
qconfig = quant.get_default_qconfig(backend)
static_model.qconfig = qconfig

# Keep embeddings and LayerNorm in floating point
for module in static_model.modules():
    if isinstance(module, (nn.Embedding, nn.LayerNorm)):
        module.qconfig = None

# Wrap all Linear layers
def wrap_linear_layers(module):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            wrapped = StaticQuantLinear(child)
            wrapped.qconfig = qconfig
            setattr(module, name, wrapped)
        else:
            wrap_linear_layers(child)

wrap_linear_layers(static_model)

# Prepare model
prepared_model = quant.prepare(static_model, inplace=False)
prepared_model.eval()

# Calibration with representative dialogue data
with torch.no_grad():
    for start in range(0, min(len(train_data) - 32, 20 * 32), 32):
        calibration_input = train_data[start:start+32]
        calibration_input = calibration_input.unsqueeze(0).cpu()
        prepared_model(calibration_input)

# Convert to static quantized model
static_quantized_model = quant.convert(
    prepared_model,
    inplace=False
)

static_quantized_model.eval()

print("Static quantization completed!")

# Save model
torch.save(
    static_quantized_model.state_dict(),
    "game_gpt_static_quantized.pth"
)

print("Static quantized model saved successfully!")
[]


In [ ]:

# Test Static Quantized GPT

test_input = torch.zeros((1, 32), dtype=torch.long)

try:
    static_quantized_model.eval()

    with torch.inference_mode():
        output = static_quantized_model(test_input)

    print("Static GPT inference successful!")
    print("Output shape:", output[0].shape)

except Exception as e:
    print("Static inference failed!")
    print("Error:", str(e))


In [ ]:

# Save dynamically quantized model

torch.save(
    quantized_dynamic_model.state_dict(),
    "game_gpt_dynamic_quantized.pth"
)

print("Quantized model saved successfully!")


In [ ]:

# Compare model sizes

quantized_size = (
    os.path.getsize("game_gpt_dynamic_quantized.pth") / 1024
)

print("Original Model Size:", round(original_size, 2), "KB")
print("Quantized Model Size:", round(quantized_size, 2), "KB")

reduction = (
    (original_size - quantized_size) / original_size
) * 100

print("Size Reduction:", round(reduction, 2), "%")


In [ ]:

import os
import pandas as pd

# Get model sizes in KB
original_size = os.path.getsize(
    "game_gpt_model.pth"
) / 1024

dynamic_size = os.path.getsize(
    "game_gpt_dynamic_quantized.pth"
) / 1024

# Save static quantized model
torch.save(
    static_quantized_model.state_dict(),
    "game_gpt_static_quantized.pth"
)

static_size = os.path.getsize(
    "game_gpt_static_quantized.pth"
) / 1024

# Comparison table
results = pd.DataFrame({
    "Model": [
        "Original GPT",
        "Dynamic Quantized GPT",
        "Static Quantized GPT"
    ],
    "Size (KB)": [
        original_size,
        dynamic_size,
        static_size
    ]
})

print(results.to_string(index=False))


In [ ]:
# Plot model-size comparison

models = ["Original GPT", "Dynamic GPT", "Static GPT"]
sizes = [original_size, dynamic_size, static_size]

plt.figure(figsize=(8, 5))
plt.bar(models, sizes)
plt.ylabel("Model Size (KB)")
plt.title("GPT Model Size Comparison")
plt.tight_layout()
plt.show()


In [ ]:

# FINAL: Fair Inference Time Comparison

import time
import torch
import pandas as pd

test_input = torch.zeros((1, 32), dtype=torch.long)

def benchmark(model, runs=200, warmup=20):
    model.eval()

    with torch.inference_mode():

        # Warm-up
        for _ in range(warmup):
            model(test_input)

        timings = []

        # Repeated measurements
        for _ in range(5):
            start = time.perf_counter()

            for _ in range(runs):
                model(test_input)

            end = time.perf_counter()

            timings.append((end - start) / runs * 1000)

    return sum(timings) / len(timings)

# Benchmark all three models
original_time = benchmark(model_cpu)
dynamic_time = benchmark(quantized_dynamic_model)
static_time = benchmark(static_quantized_model)

# Display comparison
results = pd.DataFrame({
    "Model": [
        "Original GPT",
        "Dynamic Quantized GPT",
        "Static Quantized GPT"
    ],
    "Average Inference Time (ms)": [
        original_time,
        dynamic_time,
        static_time
    ]
})

print(results.to_string(index=False))
[]